In [ ]:
!pip install "transformers[torch]"

In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration
import torch

In [ ]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [ ]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [ ]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [ ]:
train_data.sample(10)

,id,dialogue,summary
12054,13818517,Lulu: Hey chickie\r\nLulu: can you do me a fav...,Lulu still hasn't get paid for her last gig an...
631,13829305,"Tom: Did we register after all?\r\nPaul: Lol, ...","Tom and Paul registered for a marathon, but th..."
11092,13863195,Gabrielle: Can you clean the kitchen?\nGabriel...,Gabrielle's friends from the office are coming...
14583,13681492,Rufus: Guess what Martha did\r\nStella: She sa...,Martha left a rude note for Rufus so he talked...
6023,13681686,Melinda: How's your game?\r\nTeresa: u mean Di...,Teresa hasn't played her Dixit yet because she...
884,13680669,Gemma: Actually I can't breath\r\nArmando: So ...,Gemma has trouble breathing. She has already s...
3751,13810094,"Nora: You're back in town?\r\nBen: Yes, just l...","Ben has just landed back in town, is at the of..."
3700,13728616,Catherine: Can you send me this article about ...,Pauline and Catherine find the article about a...
9440,13728635,Jen: Fuck thse fucken fucks! Looks like I'm ou...,Jen has been fired again and she's angry about...
9527,13828095,Paul: coffee break?\r\nJane: and a cigarette?\...,Paul and Jane will have a coffee break and a c...


In [ ]:
train_data.shape

(14732, 3)

In [ ]:
val_data.shape

(818, 3)

In [ ]:
# random sampling
train_data = train_data.sample(n= 4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n =500, random_state=42).reset_index(drop= True)

In [ ]:
train_data.shape

(4000, 3)

In [ ]:
val_data.shape

(500, 3)

Data Pre-processing

In [ ]:
import re
def clean_data(text):
  text = re.sub(r"\r\n", " ", text) # replace carriage returns and newlines with space
  text = re.sub(r"\s+", " ", text)  # fix \5+ to \s+ for matching spaces
  text = re.sub(r"<.*?>", " ", text) # fix text.sub to re.sub
  text = text.strip().lower()
  return text

In [ ]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

Tokenizer

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
# raw data => tokenized inputs for fine_tunning
def tokenize(data):
  inputs = tokenizer(data["dialogue"], padding = "max_length", truncation= True, max_length= 512)
  targets = tokenizer(data["summary"], padding = "max_length", truncation= True, max_length= 150)

  inputs["targets"] = targets["input_ids"] # token_ids => Add to input as labels
  return inputs

In [ ]:
# 1. Install sentencepiece which is required for T5Tokenizer
!pip install sentencepiece

# 2. Re-import and initialize the tokenizer
from transformers import T5Tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-small")

# 3. Apply the tokenization
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()
print("Tokenization completed successfully!")

Tokenization completed successfully!


Working with our model

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
# fine_tune
import torch
if torch.backends.mps.is_available():
  device  =torch.device("mps")
elif torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")
print("device", device)
model.to(device)

device cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
# training arguments
training_args = TrainingArguments(
    output_dir = "/results",

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy="epoch",
    save_strategy = "epoch",

    warmup_steps = 500
)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [ ]:
# 1. Fix dataset keys by renaming 'targets' to 'labels' and formatting them as standard dictionary items
def format_dataset(dataset_list):
    formatted = []
    for item in dataset_list:
        formatted.append({
            "input_ids": item["input_ids"],
            "attention_mask": item["attention_mask"],
            "labels": item["targets"] # Rename 'targets' to 'labels'
        })
    return formatted

formatted_train = format_dataset(train_dataset)
formatted_val = format_dataset(val_dataset)

# 2. Re-initialize the Trainer with corrected datasets
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=formatted_train,
    eval_dataset=formatted_val
)

# 3. Train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.627975,0.381347
2,0.397496,0.360192
3,0.374146,0.354347
4,0.362841,0.350292
5,0.355288,0.349880
6,0.351331,0.349236


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9115128529866536, metrics={'train_runtime': 1343.4386, 'train_samples_per_second': 17.865, 'train_steps_per_second': 2.233, 'total_flos': 3248203235328000.0, 'train_loss': 0.9115128529866536, 'epoch': 6.0})

In [ ]:
# model load => fine_tuned=> save the model

In [ ]:
model.save_pretrained("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./fine_tuned_model/tokenizer_config.json',
 './fine_tuned_model/tokenizer.json')

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("./fine_tuned_model")
tokenizer= T5Tokenizer.from_pretrained("./fine_tuned_model")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
import os

model_dir = "./fine_tuned_model"
if os.path.exists(model_dir):
    print(f"✅ Directory '{model_dir}' exists!")
    print("Files inside:", os.listdir(model_dir))
else:
    print(f"❌ Directory '{model_dir}' was not found.")

✅ Directory './fine_tuned_model' exists!
Files inside: ['model.safetensors', 'tokenizer.json', 'generation_config.json', 'tokenizer_config.json', 'config.json']


In [ ]:
import os

# Let's search the filesystem to find where the model files actually are
found_paths = []
for root, dirs, files in os.walk('/'):
    # Skip system directories to search faster
    if any(p in root for p in ['/proc', '/sys', '/usr', '/lib', '/var', '/bin', '/sbin', '/etc', '/boot', '/dev']):
        continue
    if 'config.json' in files and ('model.safetensors' in files or 'pytorch_model.bin' in files):
        found_paths.append(root)

if found_paths:
    print("✅ Found your saved model in these locations:")
    for path in found_paths:
        print(f"- {path}")
        print("  Files:", os.listdir(path))
else:
    print("❌ Could not find model files in the common directories. Let's list the current folder structure:")
    print("Current working directory:", os.getcwd())
    print("Root content:", os.listdir('.'))
    if os.path.exists('/results'):
        print("Results content:", os.listdir('/results'))

✅ Found your saved model in these locations:
- /root/.cache/huggingface/hub/models--t5-small/snapshots/df1b051c49625cf57a3d0d8d3863ed4d13564fe4
  Files: ['model.safetensors', 'tokenizer.json', 'generation_config.json', 'tokenizer_config.json', 'config.json', 'spiece.model']
- /results/checkpoint-1500
  Files: ['scheduler.pt', 'model.safetensors', 'rng_state.pth', 'generation_config.json', 'training_args.bin', 'config.json', 'optimizer.pt', 'trainer_state.json']
- /results/checkpoint-500
  Files: ['scheduler.pt', 'model.safetensors', 'rng_state.pth', 'generation_config.json', 'training_args.bin', 'config.json', 'optimizer.pt', 'trainer_state.json']
- /results/checkpoint-3000
  Files: ['scheduler.pt', 'model.safetensors', 'rng_state.pth', 'generation_config.json', 'training_args.bin', 'config.json', 'optimizer.pt', 'trainer_state.json']
- /results/checkpoint-2500
  Files: ['scheduler.pt', 'model.safetensors', 'rng_state.pth', 'generation_config.json', 'training_args.bin', 'config.json', 

Test the core logic for summerization

In [ ]:
def summerize_dialogue(dialogue):
  dialogue = clean_data(dialogue)
  #tokenize
  inputs = tokenizer(
      dialogue,
      max_length = 512,
      padding = "max_length",
      truncation = True,
      return_tensors = "pt"
  ).to(device)

  # generate the summary => token ids
  model.to(device)
  targets = model.generate(
      input_ids = inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_length = 150,
      num_beams = 4,
      early_stopping = True
  )
  # token ids convert to summary => decoding (decoded our output)
  summary = tokenizer.decode(targets[0], skip_special_tokens = True)
  return summary

In [ ]:
# Define a test dialogue to summarize
test_dialogue = """Hannah: Are you coming to the party tonight?
Mark: I would love to, but I have to study for my final exam tomorrow.
Hannah: That's a bummer! Let me know if you change your mind."""

# Call the function (colon removed from the end of this line)
summary = summerize_dialogue(test_dialogue)
print("Summary: ", summary)

Summary:  mark is coming to the party tonight. hannah will study for her final exam tomorrow.


In [ ]:
import shutil
from google.colab import files

# Define the folder to download and the output zip file path
folder_to_zip = './fine_tuned_model'
output_zip = './fine_tuned_model.zip'

print("⚡ Compressing your fine-tuned model folder... This might take a few moments.")
shutil.make_archive('./fine_tuned_model', 'zip', folder_to_zip)

print("📥 Starting download of fine_tuned_model.zip...")
files.download(output_zip)

⚡ Compressing your fine-tuned model folder... This might take a few moments.
📥 Starting download of fine_tuned_model.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>